In [1]:
import pandas as pd 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import pickle
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import joblib

In [2]:
data = pd.read_csv("cleaned_data.csv")

In [3]:
data.head()

,datetime,machineID,volt,rotate,pressure,vibration,model,age,target,error_count_24h,maintenance_count_30d
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18,0,0.0,0.0
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18,0,0.0,0.0
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18,0,0.0,0.0
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18,0,0.0,0.0
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18,0,0.0,0.0


In [4]:
split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
for train_index,test_index in split.split(data,data["target"]):
    train_data = data.loc[train_index]
    test_data = data.loc[test_index]


In [5]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 700880 entries, 526895 to 798941
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   datetime               700880 non-null  object 
 1   machineID              700880 non-null  int64  
 2   volt                   700880 non-null  float64
 3   rotate                 700880 non-null  float64
 4   pressure               700880 non-null  float64
 5   vibration              700880 non-null  float64
 6   model                  700880 non-null  object 
 7   age                    700880 non-null  int64  
 8   target                 700880 non-null  int64  
 9   error_count_24h        700880 non-null  float64
 10  maintenance_count_30d  700880 non-null  float64
dtypes: float64(6), int64(3), object(2)
memory usage: 64.2+ MB


In [6]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 175220 entries, 234177 to 817295
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   datetime               175220 non-null  object 
 1   machineID              175220 non-null  int64  
 2   volt                   175220 non-null  float64
 3   rotate                 175220 non-null  float64
 4   pressure               175220 non-null  float64
 5   vibration              175220 non-null  float64
 6   model                  175220 non-null  object 
 7   age                    175220 non-null  int64  
 8   target                 175220 non-null  int64  
 9   error_count_24h        175220 non-null  float64
 10  maintenance_count_30d  175220 non-null  float64
dtypes: float64(6), int64(3), object(2)
memory usage: 16.0+ MB


In [7]:
train_data["target"].value_counts()

target
0    687133
1     13747
Name: count, dtype: int64

In [8]:
test_data["target"].value_counts()

target
0    171783
1      3437
Name: count, dtype: int64

In [9]:
train_data_label = train_data["target"]
train_data.drop(["target","datetime","machineID"],axis=1,inplace=True)

In [10]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 700880 entries, 526895 to 798941
Data columns (total 8 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   volt                   700880 non-null  float64
 1   rotate                 700880 non-null  float64
 2   pressure               700880 non-null  float64
 3   vibration              700880 non-null  float64
 4   model                  700880 non-null  object 
 5   age                    700880 non-null  int64  
 6   error_count_24h        700880 non-null  float64
 7   maintenance_count_30d  700880 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 48.1+ MB


In [11]:
test_data_label = test_data["target"]
test_data.drop(["target","datetime","machineID"],axis=1,inplace=True)

In [12]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 175220 entries, 234177 to 817295
Data columns (total 8 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   volt                   175220 non-null  float64
 1   rotate                 175220 non-null  float64
 2   pressure               175220 non-null  float64
 3   vibration              175220 non-null  float64
 4   model                  175220 non-null  object 
 5   age                    175220 non-null  int64  
 6   error_count_24h        175220 non-null  float64
 7   maintenance_count_30d  175220 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 12.0+ MB


In [13]:
cat_features = train_data.select_dtypes(include="object").columns.tolist()
num_features = train_data.select_dtypes(exclude="object").columns.tolist()

In [14]:
cat_features

['model']

In [15]:
num_features

['volt',
 'rotate',
 'pressure',
 'vibration',
 'age',
 'error_count_24h',
 'maintenance_count_30d']

In [16]:
#Pipeline 

num_pipeline = Pipeline([
    ("num",StandardScaler())  
])

cat_pipeline = Pipeline([
    ("cat",OneHotEncoder())
])

general_pipeline = ColumnTransformer([
    ("num_pipe",num_pipeline,num_features),
    ("cat_pipe",cat_pipeline,cat_features)
])

In [17]:
train_processed_data = general_pipeline.fit_transform(train_data)
test_processed_data = general_pipeline.transform(test_data)

In [18]:
#Logistic Regressiom

model1 = LogisticRegression(class_weight="balanced",random_state=42)
model1.fit(train_processed_data,train_data_label)
pred1 = model1.predict(test_processed_data)

In [19]:
#Random Forest 

model2 = RandomForestClassifier(
    n_estimators=50,   # default is 100, try 50
    max_depth=15,      # add this to limit tree depth
    random_state=42
)
model2.fit(train_processed_data,train_data_label)
pred2 = model2.predict(test_processed_data)

In [20]:
#Xgboost

model3 = XGBClassifier(random_state=42,eval_metric="logloss")
model3.fit(train_processed_data,train_data_label)
pred3 = model3.predict(test_processed_data)

In [21]:
#Accuracy Score 
print(accuracy_score(pred1,test_data_label))
print(accuracy_score(pred2,test_data_label))
print(accuracy_score(pred3,test_data_label))

0.9356979796826846
0.9885229996575733
0.9878267321082068


In [22]:
# Classification report 
print(classification_report(pred1,test_data_label))
print(classification_report(pred2,test_data_label))
print(classification_report(pred3,test_data_label))

              precision    recall  f1-score   support

           0       0.93      1.00      0.97    160692
           1       0.97      0.23      0.37     14528

    accuracy                           0.94    175220
   macro avg       0.95      0.61      0.67    175220
weighted avg       0.94      0.94      0.92    175220

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    172776
           1       0.56      0.79      0.66      2444

    accuracy                           0.99    175220
   macro avg       0.78      0.89      0.83    175220
weighted avg       0.99      0.99      0.99    175220

              precision    recall  f1-score   support

           0       1.00      0.99      0.99    172736
           1       0.55      0.76      0.64      2484

    accuracy                           0.99    175220
   macro avg       0.77      0.88      0.82    175220
weighted avg       0.99      0.99      0.99    175220



In [23]:
joblib.dump(model2, "model.pkl")
joblib.dump(general_pipeline, "pipeline.pkl")

['pipeline.pkl']